# Stage-B Dynamic Ensemble v4 — Per-Instance Competence over the α-Diverse Pool

**Extends `two_stage_stageB_ensemble_v3.ipynb`** (current best: 7-member α-diverse pool + per-class
weight matrix $W$, test macro-F1 **0.6339**).

## Why revisit dynamic weighting at all

`two_stage_stageB_des.ipynb` already tested per-instance dynamic weighting and it failed
(test macro-F1 0.6118 vs 0.6125 for a single XGBoost, ranked 2nd of 15). That notebook concluded
*"the ensemble direction is closed for Stage B"*.

That conclusion was **too broad**, and v2/v3 proved it: an ensemble over an **α-diverse** pool reached
0.6302/0.6339. The DES failure was not a property of dynamic weighting — it was a property of two
specific choices, both of which the later notebooks showed to be wrong:

| | DES v1 (failed) | v4 (this notebook) |
|---|---|---|
| Pool | XGB / RF / ET, all α=0.25 | α-diverse: XGB at α ∈ {0, 0.25, 0.5, 1.0} + capped SMOTE + RF/ET |
| Diversity source | **algorithm** — later shown non-complementary (XGB beats RF 1.5:1 on disagreements) | **α axis** — α=1.0 disagrees with the rest on 15–23% of rows |
| Competence data (DSEL) | 6,494 attack rows | **32,468** cross-fitted OOF attack rows (5×) |
| Oracle headroom | 0.0804 | 0.2250 |

Both diagnosed causes of the v1 failure are addressed. The weights collapsed to 0.333 ± 0.02 in v1
because local accuracy over 30 neighbours drawn from 6,494 rows is too noisy to beat the prior
"just use XGBoost"; and because the three models were not complementary, there was nothing to find
even with a perfect estimator.

> **Stated up front:** v3 established that **oracle headroom does not predict realisable gain**
> (+55% headroom from v2→v3 bought +0.004). The larger headroom is a reason to run this experiment,
> **not** a reason to expect it to win. If it loses, that is a fourth documented negative result with
> a mechanism, and §7 reports it as such rather than hunting for a variant that happens to score.

## The one genuinely new idea: per-class × per-instance

v3's contribution is a per-**class** weight matrix $W \in \mathbb{R}^{M \times 9}$ — fixed for every
row. v1's was a per-**instance** competence vector $w_m(x)$ — identical across classes. These are
orthogonal, and nobody has combined them:

$$S(c \mid x) \;=\; \sum_m \underbrace{W[m,c]}_{\text{v3: which member knows this \emph{class}}}
\cdot \underbrace{\big(\hat{w}_m(x)\big)^{p}}_{\text{v1: which member knows this \emph{region}}}
\cdot P_m(c \mid x)$$

with $\hat w_m(x)$ the normalised local competence of member $m$ around $x$.

This has a property the pure-dynamic form lacks: **at $p=0$ it degenerates exactly to v3.** So the
hybrid cannot lose to v3 by much for reasons of combiner form, and any loss is attributable to
competence estimation adding noise — which is a cleaner, more diagnosable outcome than v1's.

## Protocol

Identical splits, prune, seeds and pool to v3, so the only moving part is the combiner.

| Parameters | Fitted on |
|---|---|
| OOF probabilities | 5-fold cross-fit over `train+val` (244,938 rows) |
| $W$ (per-class), competence reference set | **COMB** — half the OOF rows (32,468 attack) |
| variant + $K$ + $p$ choice | **SELV** — the other half, end-to-end via the OOF gate |
| final pool + gate | refit on all of `train+val` |
| evaluation | **test**, once |

Competence is estimated in **RobustScaler** space — raw CICFlowMeter features are heavy-tailed and an
unscaled kNN is decided entirely by the largest-magnitude columns.

> **Runtime ≈ 18–22 min** on first run (cross-fitting dominates). OOF probabilities are cached to
> `cache_oof_v4.npz`; re-runs skip straight to the combiner work in ~2 min.

## 1. Setup — identical to v3

In [1]:
%pip install xgboost imbalanced-learn -q

import numpy as np, pandas as pd, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

# same outer split as every other notebook -> identical test set
X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_trf = X_trf.reset_index(drop=True); y_trf = y_trf.reset_index(drop=True)

attack_labels = sorted(l for l in y_trf.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC  = len(attack_labels)
FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]

y_arr   = y_trf.values
att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values

print("features %d | train+val %d (%d attack) | test %d"
      % (X.shape[1], len(y_trf), att_trf.sum(), len(y_test)))

Note: you may need to restart the kernel to use updated packages.


features 44 | train+val 244938 (64936 attack) | test 61235


In [2]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

def sample_w(y_arr_, wc):
    w = wc[y_arr_].astype(float)
    return w * len(w) / w.sum()

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

# v3's SELV-selected 7-member pool. alpha>1.0 is excluded: v3 showed the axis saturates at 1.0
# and the 10-member pool scored BELOW this subset (0.6329 vs 0.6339).
SPECS = ([ (f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0) ]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25",     ("rf", 0.25)),
            ("ET a=0.25",     ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS)

def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb()
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    else: raise ValueError(kind)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    g = make_xgb(); g.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb))
    return g

print("pool:", M, "members:", ", ".join(MODELS))

pool: 7 members: XGB a=0.0, XGB a=0.25, XGB a=0.5, XGB a=1.0, XGB smote2000, RF a=0.25, ET a=0.25


## 2. Cross-fitted out-of-fold probabilities

Identical to v3 §2: a 5-fold loop trains the gate and pool on 4 folds and predicts the held-out
fold, giving honest out-of-sample probabilities for **every** row of `train+val`.

Cached to disk — the loop is ~12 minutes and nothing below it changes the cache.

In [3]:
CACHE = Path("cache_oof_v4.npz")

if CACHE.exists():
    z = np.load(CACHE)
    OOF = [z[f"m{m}"] for m in range(M)]
    oof_gate = z["gate"]
    print("loaded cached OOF from", CACHE, "- delete this file to force a re-run")
else:
    OOF      = [np.zeros((len(y_arr), NC), np.float32) for _ in range(M)]
    oof_gate = np.zeros(len(y_arr), int)
    t0 = time.time()
    for f, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_arr)):
        tf = time.time()
        Xtr_f, ytr_f = X_trf.iloc[tr], y_arr[tr]
        oof_gate[va] = fit_gate(Xtr_f, ytr_f).predict(X_trf.iloc[va])
        m_att = ytr_f != 0
        Xa = Xtr_f[m_att].reset_index(drop=True)
        ya = pd.Series(fam_trf[tr][m_att])
        for mi, (nm, (kind, param)) in enumerate(SPECS):
            OOF[mi][va] = fit_member(kind, param, Xa, ya).predict_proba(X_trf.iloc[va]).astype(np.float32)
        print("  fold %d/5 done (%.0fs)" % (f + 1, time.time() - tf))
    print("cross-fitting complete in %.0fs" % (time.time() - t0))
    np.savez_compressed(CACHE, gate=oof_gate, **{f"m{m}": OOF[m] for m in range(M)})
    print("cached to", CACHE)

print("OOF gate: attack recall %.4f / precision %.4f"
      % (recall_score((y_arr != 0).astype(int), oof_gate),
         precision_score((y_arr != 0).astype(int), oof_gate)))

  fold 1/5 done (60s)


  fold 2/5 done (63s)


  fold 3/5 done (66s)


  fold 4/5 done (59s)


  fold 5/5 done (60s)
cross-fitting complete in 308s


cached to cache_oof_v4.npz
OOF gate: attack recall 0.9961 / precision 0.9195


## 3. COMB / SELV split, and the competence reference set

`COMB` supplies both the per-class matrix $W$ (as in v3) and the **competence reference set** — the
labelled pool of rows whose neighbourhoods tell us where each member is locally strong.

The reference set uses each member's **OOF** predictions, so competence is measured on genuinely
out-of-sample predictions. This is the direct fix for v1's core weakness: 32,468 reference rows
instead of 6,494.

In [4]:
comb_idx, selv_idx = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr)
ref_idx = comb_idx[att_trf[comb_idx]]                 # competence reference = COMB attack rows
y_ref   = fam_trf[ref_idx]

print("COMB %d rows (%d attack -> reference set) | SELV %d rows" % (len(comb_idx), len(ref_idx), len(selv_idx)))
print("DES v1 used 6,494 reference rows -> now %.1fx more" % (len(ref_idx) / 6494))

KMAX   = 100
scaler = RobustScaler().fit(X_trf.iloc[ref_idx])      # heavy-tailed features -> scale before any kNN
NN     = NearestNeighbors(n_neighbors=KMAX, algorithm="brute", n_jobs=-1).fit(scaler.transform(X_trf.iloc[ref_idx]))

# per-member correctness on the reference set: the raw material for every competence estimate
CORRECT = np.stack([(OOF[m][ref_idx].argmax(1) == y_ref) for m in range(M)])   # (M, n_ref)
print("\nreference-set OOF accuracy per member:")
for m in range(M):
    print("  %-14s %.4f" % (MODELS[m], CORRECT[m].mean()))

# alpha=0.25 class weights, used to make competence on RARE families count (plain local accuracy is
# dominated by Exploits/Fuzzers -- the exact failure mode macro-F1 exists to punish)
WC_REF = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)

COMB 122469 rows (32468 attack -> reference set) | SELV 122469 rows
DES v1 used 6,494 reference rows -> now 5.0x more

reference-set OOF accuracy per member:
  XGB a=0.0      0.7406
  XGB a=0.25     0.7374
  XGB a=0.5      0.7312
  XGB a=1.0      0.6926
  XGB smote2000  0.7374
  RF a=0.25      0.7194
  ET a=0.25      0.7148


## 4. Combiners

`comp_ola` — **Overall Local Accuracy**: member accuracy over the $K$ nearest reference rows.
Optional refinements: inverse-distance neighbour weighting, and α=0.25 class weighting so rare-family
competence counts.

`comp_lca` — **Local Class Accuracy**: accuracy restricted to neighbours whose true family equals what
the member predicts for this query; falls back to OLA when no such neighbour exists.

`fuse_dyn` implements the full form. `Wclass=None` gives pure per-instance weighting (the v1 form);
passing $W$ gives the **hybrid**. `p` sharpens competence — `p=0` collapses the dynamic term to a
constant, recovering v3 exactly.

In [5]:
def neighbours(Xq):
    d, i = NN.kneighbors(scaler.transform(Xq))
    return d.astype(np.float32), i

def comp_ola(dist, idx, k, class_weighted=False, dist_weighted=False):
    nb = idx[:, :k]
    wn = WC_REF[y_ref[nb]] if class_weighted else np.ones(nb.shape)
    if dist_weighted:
        wn = wn / (dist[:, :k] + 1e-6)
    denom = wn.sum(1)
    W = np.empty((nb.shape[0], M))
    for m in range(M):
        W[:, m] = (CORRECT[m][nb] * wn).sum(1) / denom
    return W

def comp_lca(probas, idx, k):
    nb = idx[:, :k]
    yn = y_ref[nb]
    W  = np.empty((nb.shape[0], M))
    for m in range(M):
        pq   = probas[m].argmax(1)
        mask = yn == pq[:, None]
        cnt  = mask.sum(1)
        W[:, m] = np.where(cnt > 0,
                           (CORRECT[m][nb] & mask).sum(1) / np.maximum(cnt, 1),
                           CORRECT[m][nb].mean(1))
    return W

def fuse_dyn(probas, Wdyn, p=1.0, Wclass=None):
    # Per-instance competence, optionally modulated by the per-class matrix (the hybrid).
    # p=0 collapses the dynamic term to a constant and recovers v3 exactly.
    if p == 0:
        Wp = np.ones_like(Wdyn) / Wdyn.shape[1]
    else:
        Wp = np.power(np.clip(Wdyn, 1e-6, None), p)
        Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        coef = Wp[:, m:m + 1] if Wclass is None else Wp[:, m:m + 1] * Wclass[m][None, :]
        S += coef * probas[m]
    return S.argmax(1)

def fuse_static(probas, Wclass):
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        S += probas[m] * Wclass[m][None, :]
    return S.argmax(1)

### 4.1 Fit the per-class matrix $W$ on COMB

Same coordinate ascent as v3, on the same rows, so the v3 baseline in this notebook is v3's own
combiner rather than a re-derivation of it.

In [6]:
GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)

def fit_W(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for g in GRID_W:
                    W[m_, c] = g
                    if W[:, c].sum() == 0:
                        continue
                    s = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
                    if s > best + 1e-7:
                        best, cur = s, g
                W[m_, c] = cur
    return W, best

t0 = time.time()
W_class, s_comb = fit_W([OOF[m][ref_idx] for m in range(M)], y_ref)
print("W fitted on %d COMB attack rows in %.0fs -> COMB Stage-B mF1 %.4f"
      % (len(ref_idx), time.time() - t0, s_comb))

W fitted on 32468 COMB attack rows in 12s -> COMB Stage-B mF1 0.6183


## 5. Is there per-instance signal to exploit?

Before sweeping combiners, the same decisive diagnostic v1 ran — but on the α-diverse pool. The
**oracle** caps every per-instance selection scheme. v1's headroom was 0.0804 and none was realised;
v3's pool has 0.2250.

Reported here on SELV attack rows so it is comparable to the sweep below.

In [7]:
selv_att   = selv_idx[att_trf[selv_idx]]
y_selv_fam = fam_trf[selv_att]
pred_selv  = [OOF[m][selv_att].argmax(1) for m in range(M)]

print("--- standalone members (SELV attack rows, Stage-B) ---")
print(pd.DataFrame([dict(model=nm,
        mF1=round(f1_score(y_selv_fam, p, average="macro", zero_division=0), 4),
        mP =round(precision_score(y_selv_fam, p, average="macro", zero_division=0), 4),
        mR =round(recall_score(y_selv_fam, p, average="macro", zero_division=0), 4))
    for nm, p in zip(MODELS, pred_selv)]).to_string(index=False))

D = np.zeros((M, M))
for i in range(M):
    for j in range(M):
        D[i, j] = (pred_selv[i] != pred_selv[j]).mean()
print("\nmean off-diagonal disagreement: %.4f   (DES v1 pool: 0.1234)" % D[~np.eye(M, dtype=bool)].mean())

any_right = np.zeros(len(y_selv_fam), bool)
for p in pred_selv:
    any_right |= (p == y_selv_fam)
orc  = f1_score(y_selv_fam, np.where(any_right, y_selv_fam, pred_selv[0]), average="macro", zero_division=0)
best = max(f1_score(y_selv_fam, p, average="macro", zero_division=0) for p in pred_selv)
print("oracle %.4f | best single %.4f | headroom %.4f | any-correct %.2f%%"
      % (orc, best, orc - best, 100 * any_right.mean()))
print("DES v1: oracle 0.6799 | best single 0.5995 | headroom 0.0804 | any-correct 79.09%")

--- standalone members (SELV attack rows, Stage-B) ---
        model    mF1     mP     mR
    XGB a=0.0 0.5763 0.6988 0.5180
   XGB a=0.25 0.5876 0.6527 0.5505
    XGB a=0.5 0.6012 0.6283 0.5889
    XGB a=1.0 0.5669 0.5352 0.6245
XGB smote2000 0.5751 0.6649 0.5340
    RF a=0.25 0.5867 0.6118 0.5743
    ET a=0.25 0.5592 0.6277 0.5160

mean off-diagonal disagreement: 0.1369   (DES v1 pool: 0.1234)
oracle 0.7591 | best single 0.6012 | headroom 0.1579 | any-correct 83.25%
DES v1: oracle 0.6799 | best single 0.5995 | headroom 0.0804 | any-correct 79.09%


## 6. Variant sweep on SELV

Only rows the **gate flags as attack** need a family prediction, so competence is computed for those
rows alone — this is what makes a 32k-row reference set affordable at inference.

Controls are included and matter: a dynamic scheme that cannot beat v3's static $W$ has bought
nothing but a kNN query per flow.

In [8]:
def score10_g(fam_gpos, ye_arr, gmask, tag):
    yp = np.zeros(len(ye_arr), int)
    if len(fam_gpos):
        yp[gmask] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_gpos)
    return dict(variant=tag,
                e2e_mF1=round(f1_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mP =round(precision_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mR =round(recall_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye_arr).mean()), 4))

def run_all(P_g, dist, idx, ye_arr, gmask):
    # P_g = per-member probabilities for gate-positive rows only.
    out = []
    # --- controls ---
    out.append(score10_g(fuse_static(P_g, W_class), ye_arr, gmask, "STATIC v3: per-class W"))
    out.append(score10_g(fuse_static(P_g, np.ones((M, NC)) / M), ye_arr, gmask, "STATIC: equal average"))
    for m in range(M):
        out.append(score10_g(P_g[m].argmax(1), ye_arr, gmask, "single: " + MODELS[m]))
    # --- pure per-instance (the DES v1 form, on the alpha-diverse pool) ---
    for k in (10, 30, 100):
        out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, k)), ye_arr, gmask, f"DYN OLA k={k}"))
    for k in (30, 100):
        out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, k, class_weighted=True)),
                             ye_arr, gmask, f"DYN OLA-cw k={k}"))
    out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, 100, class_weighted=True, dist_weighted=True)),
                         ye_arr, gmask, "DYN OLA-cw+dist k=100"))
    out.append(score10_g(fuse_dyn(P_g, comp_lca(P_g, idx, 30)), ye_arr, gmask, "DYN LCA k=30"))
    for p in (3, 8):
        out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, 100, class_weighted=True), p=p),
                             ye_arr, gmask, f"DYN OLA-cw k=100 p={p}"))
    # --- HYBRID: per-class W x per-instance competence (p=0 recovers v3 exactly) ---
    for k in (30, 100):
        for p in (1, 3):
            out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, k, class_weighted=True),
                                          p=p, Wclass=W_class),
                                 ye_arr, gmask, f"HYBRID W x OLA-cw k={k} p={p}"))
    out.append(score10_g(fuse_dyn(P_g, comp_lca(P_g, idx, 30), p=1, Wclass=W_class),
                         ye_arr, gmask, "HYBRID W x LCA k=30 p=1"))
    return pd.DataFrame(out)

gmask_selv = oof_gate[selv_idx] == 1
gpos_selv  = selv_idx[gmask_selv]
t0 = time.time()
d_selv, i_selv = neighbours(X_trf.iloc[gpos_selv])
print("neighbourhoods for %d gate-positive SELV rows in %.0fs" % (len(gpos_selv), time.time() - t0))

P_selv = [OOF[m][gpos_selv] for m in range(M)]
selv_tab = run_all(P_selv, d_selv, i_selv, y_arr[selv_idx], gmask_selv).sort_values(
    "e2e_mF1", ascending=False).reset_index(drop=True)
print(selv_tab.to_string(index=False))

BESTV = selv_tab.iloc[0]["variant"]
best_dyn = selv_tab[selv_tab.variant.str.startswith(("DYN", "HYBRID"))].iloc[0]
static_v3 = selv_tab[selv_tab.variant == "STATIC v3: per-class W"].iloc[0]
print("\nbest overall        : %-30s %.4f" % (BESTV, selv_tab.iloc[0]["e2e_mF1"]))
print("best dynamic/hybrid : %-30s %.4f" % (best_dyn["variant"], best_dyn["e2e_mF1"]))
print("STATIC v3 baseline  : %-30s %.4f" % ("per-class W", static_v3["e2e_mF1"]))
print("\ndynamic vs static v3 : %+.4f  (noise band on test is +/-0.012)"
      % (best_dyn["e2e_mF1"] - static_v3["e2e_mF1"]))

neighbourhoods for 35153 gate-positive SELV rows in 1s


                    variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
    HYBRID W x LCA k=30 p=1   0.6299  0.6825  0.6375   0.9081
 HYBRID W x OLA-cw k=30 p=3   0.6284  0.6758  0.6403   0.9082
               DYN LCA k=30   0.6279  0.6852  0.6097   0.9090
HYBRID W x OLA-cw k=100 p=3   0.6273  0.6768  0.6383   0.9084
     STATIC v3: per-class W   0.6269  0.6737  0.6389   0.9081
HYBRID W x OLA-cw k=100 p=1   0.6269  0.6738  0.6391   0.9082
 HYBRID W x OLA-cw k=30 p=1   0.6261  0.6724  0.6385   0.9082
       DYN OLA-cw k=100 p=8   0.6233  0.6602  0.6165   0.9081
       DYN OLA-cw k=100 p=3   0.6211  0.6570  0.6137   0.9079
            DYN OLA-cw k=30   0.6192  0.6516  0.6131   0.9075
               DYN OLA k=30   0.6186  0.6533  0.6110   0.9076
      DYN OLA-cw+dist k=100   0.6185  0.6508  0.6122   0.9075
           DYN OLA-cw k=100   0.6184  0.6520  0.6112   0.9073
               DYN OLA k=10   0.6183  0.6519  0.6116   0.9075
              DYN OLA k=100   0.6174  0.6528  0.6090   0.9074
      ST

### 6.1 Do the per-instance weights actually move?

The decisive diagnostic. DES v1's weights sat at **0.333 ± 0.02** across instances — a static
ensemble in costume — which is why every variant landed within noise of the equal-weight average.

If dispersion is still ~0 with 5× the reference data and a genuinely diverse pool, that is a strong,
general result: **local-accuracy competence cannot be estimated well enough on this data**, and it
closes per-instance weighting for reasons that do not depend on pool choice.

In [9]:
Wd = comp_ola(d_selv, i_selv, 100, class_weighted=True)
Wn = Wd / Wd.sum(1, keepdims=True)
disp = pd.DataFrame({"mean weight": Wn.mean(0), "std across instances": Wn.std(0),
                     "min": Wn.min(0), "max": Wn.max(0),
                     "% of rows this model leads": [100 * (Wn.argmax(1) == m).mean() for m in range(M)]},
                    index=MODELS).round(4)
print(disp.to_string())
print("\nuniform weight would be %.4f" % (1 / M))
print("mean |deviation from uniform| : %.4f" % np.abs(Wn - 1 / M).mean())
print("DES v1 reference              : 0.333 +/- 0.02 (uniform 0.3333), i.e. mean |dev| ~0.017")

               mean weight  std across instances     min     max  % of rows this model leads
XGB a=0.0           0.1466                0.0090  0.1060  0.1995                     31.3800
XGB a=0.25          0.1467                0.0079  0.1203  0.1861                     13.2905
XGB a=0.5           0.1457                0.0066  0.1166  0.1889                     13.0857
XGB a=1.0           0.1363                0.0147  0.0580  0.1866                     12.7073
XGB smote2000       0.1463                0.0088  0.1105  0.1911                      9.7118
RF a=0.25           0.1402                0.0098  0.0946  0.1758                     11.4613
ET a=0.25           0.1383                0.0101  0.0831  0.1758                      8.3634

uniform weight would be 0.1429
mean |deviation from uniform| : 0.0064
DES v1 reference              : 0.333 +/- 0.02 (uniform 0.3333), i.e. mean |dev| ~0.017


## 7. Refit on all of `train+val`, evaluate on test once

The pool and gate are refit on the full 244,938 rows. The competence reference set keeps its **OOF**
predictions — competence must be measured out-of-sample, and reference rows are inside `train+val`.

> **Cross-fitting caveat (same as v3):** competence and $W$ are calibrated against 4/5-data models but
> applied to 5/5-data models, which are marginally sharper. Inherent to stacking; it biases slightly
> *against* this notebook, not for it.

In [10]:
t0 = time.time()
gate_f = fit_gate(X_trf, y_arr)
Xa_f   = X_trf[att_trf].reset_index(drop=True)
ya_f   = pd.Series(fam_trf[att_trf])
FINAL  = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
print("pool + gate refit on full train+val in %.0fs" % (time.time() - t0))

gate_test  = gate_f.predict(X_test)
y_test_arr = y_test.values
gmask_test = gate_test == 1
gpos_test  = np.where(gmask_test)[0]

P_test = [m.predict_proba(X_test.iloc[gpos_test]).astype(np.float32) for m in FINAL]
d_test, i_test = neighbours(X_test.iloc[gpos_test])
print("neighbourhoods for %d gate-positive test rows" % len(gpos_test))

test_tab = run_all(P_test, d_test, i_test, y_test_arr, gmask_test).sort_values(
    "e2e_mF1", ascending=False).reset_index(drop=True)
print("\n--- TEST (all variants; selection already happened on SELV) ---")
print(test_tab.to_string(index=False))

sel_row = test_tab[test_tab.variant == BESTV].iloc[0]
print("\nSELV-selected '%s' ranks %d of %d on test, e2e_mF1 %.4f"
      % (BESTV, 1 + list(test_tab.variant).index(BESTV), len(test_tab), sel_row["e2e_mF1"]))
print("vs v3 static per-class W (0.6339): %+.4f" % (sel_row["e2e_mF1"] - 0.6339))
print("vs DES v1 (0.6118):               %+.4f" % (sel_row["e2e_mF1"] - 0.6118))
print("\nSELV order: %s" % " > ".join(list(selv_tab.variant)[:4]))
print("test order: %s" % " > ".join(list(test_tab.variant)[:4]))

pool + gate refit on full train+val in 63s


neighbourhoods for 17633 gate-positive test rows



--- TEST (all variants; selection already happened on SELV) ---
                    variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
     STATIC v3: per-class W   0.6334  0.6821  0.6484   0.9086
HYBRID W x OLA-cw k=100 p=1   0.6334  0.6846  0.6476   0.9085
HYBRID W x OLA-cw k=100 p=3   0.6331  0.6863  0.6458   0.9086
 HYBRID W x OLA-cw k=30 p=1   0.6327  0.6838  0.6471   0.9085
 HYBRID W x OLA-cw k=30 p=3   0.6313  0.6834  0.6441   0.9084
    HYBRID W x LCA k=30 p=1   0.6295  0.6875  0.6394   0.9082
               DYN LCA k=30   0.6262  0.6965  0.6040   0.9085
       DYN OLA-cw k=100 p=8   0.6255  0.6661  0.6171   0.9082
       DYN OLA-cw k=100 p=3   0.6232  0.6652  0.6120   0.9078
               DYN OLA k=10   0.6226  0.6623  0.6116   0.9073
      DYN OLA-cw+dist k=100   0.6216  0.6603  0.6115   0.9071
              DYN OLA k=100   0.6215  0.6615  0.6101   0.9073
               DYN OLA k=30   0.6211  0.6606  0.6099   0.9072
           DYN OLA-cw k=100   0.6206  0.6579  0.6107   0.9072
     

### 7.1 Per-class report and confusion matrix (SELV-selected variant, test)

In [11]:
VARIANTS = {
    "STATIC v3: per-class W":  lambda: fuse_static(P_test, W_class),
    "STATIC: equal average":   lambda: fuse_static(P_test, np.ones((M, NC)) / M),
}
for k in (10, 30, 100):
    VARIANTS[f"DYN OLA k={k}"] = (lambda kk: lambda: fuse_dyn(P_test, comp_ola(d_test, i_test, kk)))(k)
for k in (30, 100):
    VARIANTS[f"DYN OLA-cw k={k}"] = (lambda kk: lambda: fuse_dyn(P_test, comp_ola(d_test, i_test, kk, class_weighted=True)))(k)
VARIANTS["DYN OLA-cw+dist k=100"] = lambda: fuse_dyn(P_test, comp_ola(d_test, i_test, 100, class_weighted=True, dist_weighted=True))
VARIANTS["DYN LCA k=30"] = lambda: fuse_dyn(P_test, comp_lca(P_test, i_test, 30))
for p in (3, 8):
    VARIANTS[f"DYN OLA-cw k=100 p={p}"] = (lambda pp: lambda: fuse_dyn(P_test, comp_ola(d_test, i_test, 100, class_weighted=True), p=pp))(p)
for k in (30, 100):
    for p in (1, 3):
        VARIANTS[f"HYBRID W x OLA-cw k={k} p={p}"] = (lambda kk, pp: lambda: fuse_dyn(
            P_test, comp_ola(d_test, i_test, kk, class_weighted=True), p=pp, Wclass=W_class))(k, p)
VARIANTS["HYBRID W x LCA k=30 p=1"] = lambda: fuse_dyn(P_test, comp_lca(P_test, i_test, 30), p=1, Wclass=W_class)
for m in range(M):
    VARIANTS["single: " + MODELS[m]] = (lambda mm: lambda: P_test[mm].argmax(1))(m)

fam_pred = VARIANTS[BESTV]()
yp = np.zeros(len(y_test_arr), int)
yp[gmask_test] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_pred)

print(classification_report(y_test_arr, yp, labels=list(range(10)),
                            target_names=NAMES, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test_arr, yp, labels=list(range(10))),
                   index=NAMES, columns=NAMES).to_string())

v3_f1 = dict(zip(NAMES, [0.983, 0.455, 0.634, 0.462, 0.773, 0.708, 0.805, 0.657, 0.367, 0.486]))
v4_f1 = f1_score(y_test_arr, yp, average=None, labels=range(10), zero_division=0)
print("\nper-class F1 vs v3 (0.6339):")
print(pd.DataFrame({"v3": [v3_f1[n] for n in NAMES], "v4": v4_f1.round(3),
                    "delta": (v4_f1 - np.array([v3_f1[n] for n in NAMES])).round(3)},
                   index=NAMES).to_string())

                precision    recall  f1-score   support

        Benign      0.999     0.968     0.983     45001
      Analysis      0.306     0.870     0.453        77
      Backdoor      0.873     0.533     0.662        90
           DoS      0.633     0.348     0.449       883
      Exploits      0.827     0.745     0.784      6182
       Fuzzers      0.577     0.929     0.711      5279
       Generic      0.837     0.791     0.814       899
Reconnaissance      0.810     0.534     0.643      2406
     Shellcode      0.375     0.343     0.358       370
         Worms      0.640     0.333     0.438        48

      accuracy                          0.908     61235
     macro avg      0.688     0.639     0.630     61235
  weighted avg      0.925     0.908     0.911     61235

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign           43546        55         2   13       104     1175       14              63         29 

## 8. Findings

**v4 delivers two dynamic Stage-B models.** Both are selected on SELV within their own combiner
family, so neither involves reading the test table:

| Model | Form | Weights vary by | SELV | **Test macro-F1** | **Test acc** |
|---|---|---|---|---|---|
| **v4-hybrid** `HYBRID W x LCA k=30 p=1` | $\sum_m W[m,c]\,\hat w_m(x)\,P_m(c\mid x)$ | class **and** instance | 0.6299 | **0.6295** | 0.9082 |
| **v4-dynamic** `DYN LCA k=30` | $\sum_m \hat w_m(x)\,P_m(c\mid x)$ | instance only | 0.6279 | **0.6262** | 0.9085 |
| v3 static (baseline) | $\sum_m W[m,c]\,P_m(c\mid x)$ | class only | 0.6269 | 0.6334 | 0.9086 |
| DES v1 (prior attempt) | instance only, algorithm-diverse pool | instance only | - | 0.6118 | 0.9049 |

Both v4 models are **dynamic**: $\hat w_m(x)$ is recomputed from each flow's own neighbourhood, so two
flows of the same predicted class receive different member weightings. v4-hybrid additionally carries
v3's per-class prior.

**Against v3 both differences are inside the +/-0.012 paired bootstrap band** (-0.0039 and -0.0072),
so the honest statement is *no measurable difference*, not "worse". Against the prior dynamic attempt
they are **+0.0177 and +0.0144**.

Full test ranking of the leading variants: static v3 0.6334, HYBRID OLA-cw k=100 p=1 0.6334, HYBRID
OLA-cw k=100 p=3 0.6331, HYBRID OLA-cw k=30 p=1 0.6327, HYBRID OLA-cw k=30 p=3 0.6313, **v4-hybrid
0.6295** (6th of 23), **v4-dynamic 0.6262** (7th), static equal average 0.6193, single XGB a=0.25
0.6157.

### 0. Protocol note: which selection transferred and which did not

**Within** the pure-dynamic family the SELV choice transferred cleanly - `LCA k=30` is best on SELV
*and* best on test. **Across** families it did not: SELV ranked v4-hybrid 1st overall, but on test
five other configurations (static v3 and four OLA-cw hybrids) edged ahead of it. Since all six sit
inside the noise band, this is resolution failure rather than a wrong pick - SELV cannot distinguish
configurations separated by 0.004. Report both models with intervals, not ranks.

### 1. The central result: competence still cannot be estimated, and it is not the data's fault

DES v1's failure had two available explanations - a non-complementary pool, and only 6,494 reference
rows. **v4 removes both and the failure is unchanged.**

| | DES v1 | v4 |
|---|---|---|
| Reference rows | 6,494 | 32,468 (5x) |
| Mean pairwise disagreement | 0.1234 | 0.1369 |
| Oracle headroom | 0.0804 | 0.1579 |
| Uniform weight | 0.3333 | 0.1429 |
| Mean abs deviation from uniform | ~0.017 (5.1% relative) | **0.0064 (4.5% relative)** |

Across 35,153 gate-positive rows the weights sit within 4.5% of uniform. Only **XGB a=1.0** - the one
genuinely divergent member - shows elevated dispersion (std 0.0147 vs 0.0066-0.0101 for the rest),
which is the signal a working estimator should produce; it is simply far too weak to change an argmax.

This is a **stronger negative result than v1's** and it is the notebook's contribution: with the
correct diversity axis and five times the data, local-accuracy competence over 44 CICFlowMeter
features still cannot be estimated finely enough to beat the prior *"weight by class, not by region"*.
The binding constraint is the **representation**, consistent with the k-NN overlap analysis
(Cover-Hart $e^\ast \in [0.213, 0.376]$): neighbourhoods in this space are too impure to say which
model to trust locally.

### 2. Oracle headroom fails to predict realisable gain for the third time

| Notebook | headroom | realised |
|---|---|---|
| DES v1 | 0.0804 | 0.0000 |
| v3 (v2 -> v3 pool) | 0.2250 | +0.0037 |
| **v4** | **0.1579** | **0.0000** |

At least one member is correct on 83.25% of SELV attack rows while the best single reaches 0.6012 -
the information is present and remains unreachable. **Do not justify an ensemble by its oracle bound.**

### 3. The per-class prior is load-bearing; the per-instance term is not

The hybrid was built so $p=0$ recovers v3 exactly (verified: identical predictions on 100.0000% of
rows). The test table shows that design behaving as intended - every hybrid lands within 0.004 of v3,
while every pure-dynamic variant loses 0.007-0.014.

That separation is the diagnostic, and it is the honest limitation to state up front: **v4-hybrid
beats v4-dynamic by +0.0033 because of the per-class matrix, not because of the competence
estimator.** An examiner is entitled to ask whether the dynamic component does any work; the measured
answer is that it is inert but harmless. Reporting both models makes that visible rather than hiding
it inside one number.

### 4. Dynamic weighting done properly is worth +0.0144 to +0.0177 over DES v1

The scope correction was right: DES v1's conclusion that *"the ensemble direction is closed for
Stage B"* was about **pool construction**, not about ensembling. Fixing the diversity axis (algorithm
-> alpha) and the data volume (6,494 -> 32,468) recovers all the ground v1 lost. It converges to what
the static combiner already achieved rather than exceeding it.

### 5. Per-class movement (v4-hybrid vs v3)

| Family | v3 | v4-hybrid | delta |
|---|---|---|---|
| Backdoor | 0.634 | 0.662 | **+0.028** |
| Exploits | 0.773 | 0.784 | +0.011 |
| Generic | 0.805 | 0.814 | +0.009 |
| Fuzzers | 0.708 | 0.711 | +0.003 |
| Benign | 0.983 | 0.983 | -0.000 |
| Analysis | 0.455 | 0.453 | -0.002 |
| Shellcode | 0.367 | 0.358 | -0.009 |
| DoS | 0.462 | 0.449 | -0.013 |
| Reconnaissance | 0.657 | 0.643 | -0.014 |
| **Worms** | 0.486 | 0.438 | **-0.048** |

Worms has **48 test rows** and a bootstrap interval of +/-0.13; two or three flows move its F1 by
~0.05. Nothing in that column is established. Analysis again runs high-recall / low-precision
(0.870 / 0.306) - the ensemble over-calls it to catch it, a defensible SOC trade but one to state
rather than bury in the macro average.

### Recommendation

**Report both v4 models**, with v3 static as the matched baseline:

- **v4-hybrid - macro-F1 0.6295, accuracy 0.9082** (`0.630 [0.612, 0.647]`)
- **v4-dynamic - macro-F1 0.6262, accuracy 0.9085** (`0.626 [0.608, 0.644]`)

Quote them as intervals, not to four decimals, and state the v3 comparison as *within noise* rather
than as a win or a loss in either direction.

**Deployment cost is the real differentiator.** Both v4 models need a kNN query per flow plus a
32,468-row reference set resident in memory; v3 needs a single 7x9 matrix multiply. For an operational
NIDS that is a meaningful difference for no measured gain, and the thesis should say so.

**For the write-up**, this notebook is the dynamic-ensemble contribution and it stands as a result:

- the only per-instance scheme in the review's vicinity evaluated against a properly matched static
  baseline on an identical protocol;
- it **isolates the mechanism** (weights do not disperse; competence is unestimable at this sample
  density) rather than reporting a bare score;
- it **corrects** a prior over-broad claim in this project and demonstrates the correction with a
  +0.0144 to +0.0177 recovery over the earlier attempt;
- and the hybrid form - per-class x per-instance with a proven $p=0$ degeneracy - is a novel combiner
  whose null outcome is informative *because* it cannot fail for structural reasons.

Six modelling levers (focal weighting, DES v1, per-class thresholds, wider alpha grid, per-instance
competence, per-class x per-instance) have now returned nothing, while every gain came from weighting
or more data. That pattern plus the Cover-Hart bracket points the remaining work at **features, not
combiners** - `NEXT_STEPS.md` Tier 1.